In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ====== 3. Install Python Packages ======
# !pip install torch torchvision torchaudio --upgrade  # PyTorch
!pip install opencv-python matplotlib numpy pycocotools psutil Pillow
!pip install tqdm

# ====== 4. Verify GPU Availability (if using CUDA) ======
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


In [4]:
import sys
sys.path.append('/content/drive/MyDrive/CV Project 2025/segmentation')

In [5]:
import os
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone


from coco_utils import get_coco, collate_fn
from engine import train_one_epoch, evaluate

In [6]:
def get_instance_segmentation_model(num_classes):
    # load an instance segmentation model pre-trained on COCO
    backbone = resnet_fpn_backbone('resnet18', pretrained=True)
    model = MaskRCNN(backbone, num_classes)

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model


In [7]:
img_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/images'
train_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/train.json'
val_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/val.json'
test_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/test.json'

In [8]:
coco_train_ds = get_coco(img_data_path, train_nn_data_path, mode="instances", train=True)
coco_val_ds = get_coco(img_data_path, val_nn_data_path, mode="instances", train=False)
coco_test_ds = get_coco(img_data_path, test_nn_data_path, mode="instances", train=False)

loading annotations into memory...
Done (t=1.42s)
creating index...
index created!
loading annotations into memory...
Done (t=0.48s)
creating index...
index created!
loading annotations into memory...
Done (t=0.42s)
creating index...
index created!


In [9]:
train_data_loader = torch.utils.data.DataLoader(coco_train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)

val_data_loader = torch.utils.data.DataLoader(coco_val_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

test_data_loader = torch.utils.data.DataLoader(coco_test_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [10]:
num_classes = 2

# get the model using our helper function
model = get_instance_segmentation_model(num_classes)
# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# and a learning rate scheduler which decreases the learning rate by
# 10x every 3 epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# let's train it for 10 epochs
from torch.optim.lr_scheduler import StepLR

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f370

In [11]:
num_epochs = 2

for epoch in range(num_epochs):
    # train for one epoch, printing every 5 iterations
    train_one_epoch(model, optimizer, train_data_loader, device, epoch, print_freq=5)
    # update the learning rate
    lr_scheduler.step()
    # # evaluate on the test dataset
    evaluate(model, val_data_loader, device=device)

    torch.save(
      {
          "epoch": epoch,
          "model_state_dict": model.state_dict(),
          "optimizer_state_dict": optimizer.state_dict(),
      },
      "maskrcnn_resnet18_same_object.pt",
    )

/content/drive/MyDrive/CV Project 2025/segmentation/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [  0/580]  eta: 2:04:33  lr: 0.000014  loss: 5.6313 (5.6313)  loss_classifier: 0.7581 (0.7581)  loss_box_reg: 0.0497 (0.0497)  loss_mask: 4.0399 (4.0399)  loss_objectness: 0.7091 (0.7091)  loss_rpn_box_reg: 0.0745 (0.0745)  time: 12.8855  data: 10.0102  max mem: 2888
Epoch: [0]  [  5/580]  eta: 0:27:18  lr: 0.000057  loss: 6.9448 (7.7112)  loss_classifier: 0.7326 (0.7502)  loss_box_reg: 0.0497 (0.0781)  loss_mask: 5.4322 (6.1401)  loss_objectness: 0.7091 (0.7097)  loss_rpn_box_reg: 0.0186 (0.0331)  time: 2.8490  data: 1.8043  max mem: 3059
Epoch: [0]  [ 10/580]  eta: 0:21:50  lr: 0.000100  loss: 5.6313 (6.2671)  loss_classifier: 0.7132 (0.6697)  loss_box_reg: 0.0582 (0.0694)  loss_mask: 4.1402 (4.7904)  loss_objectness: 0.7077 (0.7069)  loss_rpn_box_reg: 0.0186 (0.0307)  time: 2.2993  data: 1.3885  max mem: 3059
Epoch: [0]  [ 15/580]  eta: 0:18:35  lr: 0.000143  loss: 4.9614 (5.1261)  loss_classifier: 0.5888 (0.5935)  loss_box_reg: 0.0582 (0.0660)  loss_mask: 3.5415 (3.7181

In [12]:
# download the model
from google.colab import files
files.download("maskrcnn_resnet18_same_object.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from torchvision.utils import draw_segmentation_masks, draw_bounding_boxes

from PIL import Image, ImageOps
import PIL
import numpy as np

def save_result(img, int_img, prediction):
    img = Image.fromarray(img.mul(255).permute(1, 2, 0).byte().numpy())

    output = prediction[0]
    masks, boxes = output["masks"], output["boxes"]

    detection_threshold = 0.8
    pred_scores = output["scores"].detach().cpu().numpy()
    pred_classes = [str(i) for i in output["labels"].cpu().numpy()]
    print(pred_classes)
    print(output["labels"])
    pred_bboxes = output["boxes"].detach().cpu().numpy()
    boxes = pred_bboxes[pred_scores >= detection_threshold].astype(np.int32)
    pred_classes = pred_classes[: len(boxes)]

    int_img = np.array(int_img)
    int_img = np.transpose(int_img, [2, 0, 1])
    int_img = torch.tensor(int_img, dtype=torch.uint8)

    colors = np.random.randint(0, 255, size=(len(pred_bboxes), 3))
    colors = [tuple(color) for color in colors]
    result_with_boxes = draw_bounding_boxes(
        int_img,
        boxes=torch.tensor(boxes),
        width=4,
        colors=colors,
        labels=pred_classes,
    )

    final_masks = masks > 0.5
    final_masks = final_masks.squeeze(1)
    seg_result = draw_segmentation_masks(
        result_with_boxes, final_masks, colors=colors, alpha=0.8
    )

    seg_img = Image.fromarray(seg_result.mul(255).permute(1, 2, 0).byte().numpy())

    imgs = [img, seg_img]
    min_shape = sorted([(np.sum(i.size), i.size) for i in imgs])[0][1]
    imgs_comb = np.hstack([i.resize(min_shape) for i in imgs])

    imgs_comb = Image.fromarray(imgs_comb)
    imgs_comb.save("result.jpg")

In [15]:
# test the model on the test set

evaluate(model, test_data_loader, device=device)

Test:  [  0/507]  eta: 0:07:49  model_time: 0.1742 (0.1742)  evaluator_time: 0.1401 (0.1401)  time: 0.9269  data: 0.5904  max mem: 3413
Test:  [100/507]  eta: 0:04:38  model_time: 0.1646 (0.2648)  evaluator_time: 0.2066 (0.3282)  time: 0.7475  data: 0.0464  max mem: 3413
Test:  [200/507]  eta: 0:03:34  model_time: 0.2537 (0.2670)  evaluator_time: 0.2615 (0.3382)  time: 0.8845  data: 0.0638  max mem: 3413
Test:  [300/507]  eta: 0:02:19  model_time: 0.1740 (0.2590)  evaluator_time: 0.1343 (0.3232)  time: 0.6238  data: 0.0557  max mem: 3413
Test:  [400/507]  eta: 0:01:13  model_time: 0.2275 (0.2657)  evaluator_time: 0.2170 (0.3292)  time: 0.6593  data: 0.0451  max mem: 3413
Test:  [500/507]  eta: 0:00:04  model_time: 0.1647 (0.2635)  evaluator_time: 0.1962 (0.3262)  time: 0.6251  data: 0.0465  max mem: 3413
Test:  [506/507]  eta: 0:00:00  model_time: 0.1542 (0.2625)  evaluator_time: 0.1414 (0.3244)  time: 0.5370  data: 0.0510  max mem: 3413
Test: Total time: 0:05:42 (0.6764 s / it)
Averag

In [19]:
# randomly sample one index from the test set
sample_idx = torch.randint(len(coco_test_ds), size=(1,)).item()
int_img, img, _ = coco_test_ds[sample_idx]
model.eval()
with torch.no_grad():
  prediction = model([img.to(device)])

save_result(img, int_img, prediction)

['1', '1', '1']
tensor([1, 1, 1], device='cuda:0')


In [17]:
# download the sample test image
from google.colab import files
files.download("result.jpg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>